# 🛰️ DeepSpace — Satellite Image Super-Resolution Pipeline
### Google Colab Notebook

This notebook runs the full DeepSpace pipeline on Google Colab with GPU acceleration:

1. **Setup** — clone repo, install dependencies
2. **Mount Drive** — load your data from Google Drive
3. **Train** — train the WaveletNCSNpp super-resolution model
4. **Fine-tune** *(optional)* — few-shot fine-tuning on a new region
5. **Inference** — reconstruct low-res satellite images to 256×256
6. **Evaluate** — compute PSNR / SSIM metrics
7. **Visualize** — display input / output image pairs
8. **Save** — persist checkpoints back to Drive

> ⚡ **Runtime tip:** Go to *Runtime → Change runtime type → T4 GPU* before running.

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU.")

print(f"   PyTorch: {torch.__version__}")

## Step 1 — Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL = "https://github.com/sophiasimons/DeepSpace.git"
REPO_DIR = "/content/DeepSpace"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}/decompress
print("Working directory:", os.getcwd())

In [ ]:
# Install PyTorch (CUDA 12.1) and project dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

!pip install -q \
    pytorch_wavelets \
    einops \
    lmdb \
    Pillow \
    scikit-image \
    lpips \
    imagehash \
    ninja

print("✅ Dependencies installed.")

## Step 2 — Mount Google Drive & Link Data

Your data folder should be organized in Drive as:
```
MyDrive/DeepSpace/data/
  deepgreen_16_256/
    hr_256/        ← 256×256 high-res images
    lr_16/         ← 16×16 low-res inputs
    sr_16_256/     ← bicubic upsampled (used as model input)
```
Run the cell below to mount Drive and create a symlink.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# ── Configure these paths ──────────────────────────────────────────────────
DRIVE_DATA_DIR = "/content/drive/MyDrive/DeepSpace/data"   # ← edit if needed
LOCAL_DATA_DIR = "/content/DeepSpace/decompress/data"
# ──────────────────────────────────────────────────────────────────────────

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# Create a symlink so the training scripts find the data at the expected path
symlink_target = os.path.join(LOCAL_DATA_DIR, "deepgreen_16_256")
if not os.path.exists(symlink_target):
    os.symlink(
        os.path.join(DRIVE_DATA_DIR, "deepgreen_16_256"),
        symlink_target
    )
    print(f"✅ Symlink created: {symlink_target} → Drive")
else:
    print("Symlink already exists.")

# Quick sanity check
hr_count = len(os.listdir(os.path.join(symlink_target, "hr_256")))
print(f"   hr_256 images found: {hr_count}")

## Step 3 — Train the Model

Trains the WaveletNCSNpp GAN from scratch on your dataset.  
Checkpoints are saved every epoch to `content/saved_info/`.

In [ ]:
# ── Training configuration — edit as needed ───────────────────────────────
DATASET      = "deepgreen_16_256"
DATADIR      = "./data/deepgreen_16_256"
EXP          = "deepspace_colab"
NUM_EPOCH    = 20        # increase for better quality (Colab Pro: up to 100)
BATCH_SIZE   = 6         # T4 handles 6–8 comfortably
IMAGE_SIZE   = 256
LR_SIZE      = 16
NUM_TIMESTEPS = 4
# ──────────────────────────────────────────────────────────────────────────

!python train_srwddgan.py \
    --dataset {DATASET} \
    --datadir {DATADIR} \
    --exp {EXP} \
    --num_epoch {NUM_EPOCH} \
    --batch_size {BATCH_SIZE} \
    --image_size {IMAGE_SIZE} \
    --lr_size {LR_SIZE} \
    --num_timesteps {NUM_TIMESTEPS} \
    --num_channels 3 \
    --num_channels_dae 128 \
    --n_mlp 4 \
    --ch_mult 1 1 2 2 4 4 \
    --num_res_blocks 2 \
    --attn_resolutions 16 \
    --not_use_tanh \
    --save_content \
    --save_content_every 1 \
    --save_ckpt_every 5

## Step 4 — (Optional) Few-Shot Fine-Tune on a New Region

Skip this step if you only need the base model.  
Fine-tuning adapts the model to a new geographic region using just 50–100 images.

In [ ]:
# ── Fine-tune configuration ───────────────────────────────────────────────
# Path to the foundation model checkpoint saved during training above
FOUNDATION_CKPT  = f"./content/saved_info/wddgan/{EXP}/{EXP}/netG_final.pth"

# Path to new-region data folder (must have hr_256/ and sr_16_256/ subfolders)
NEW_REGION_DIR   = "/content/drive/MyDrive/DeepSpace/data/new_region"  # ← edit

FINETUNE_EXP     = "deepspace_finetuned"
FINETUNE_EPOCHS  = 30
FINETUNE_DATA_LEN = 100   # number of images to use (50–100 recommended)
# ──────────────────────────────────────────────────────────────────────────

!python finetune_srwddgan.py \
    --foundation_ckpt {FOUNDATION_CKPT} \
    --dataset deepgreen_16_256 \
    --datadir {NEW_REGION_DIR} \
    --exp {FINETUNE_EXP} \
    --data_len {FINETUNE_DATA_LEN} \
    --num_epoch {FINETUNE_EPOCHS} \
    --batch_size 4 \
    --image_size 256 \
    --lr_size 16 \
    --num_timesteps 4 \
    --num_channels 3 \
    --num_channels_dae 128 \
    --n_mlp 4 \
    --ch_mult 1 1 2 2 4 4 \
    --num_res_blocks 2 \
    --attn_resolutions 16 \
    --not_use_tanh \
    --lr_g 5e-5 \
    --lr_d 2e-5

## Step 5 — Run Inference (Reconstruct Images)

In [ ]:
# ── Inference configuration ───────────────────────────────────────────────
# Use EXP for the base model, or FINETUNE_EXP for the fine-tuned model
INFER_EXP     = EXP
EPOCH_ID      = NUM_EPOCH   # which checkpoint epoch to load
NUM_SAMPLES   = 50          # how many test images to reconstruct
# ──────────────────────────────────────────────────────────────────────────

RESULTS_DIR = f"./content/results/{INFER_EXP}"

!python test_srwddgan.py \
    --dataset {DATASET} \
    --datadir {DATADIR} \
    --exp {INFER_EXP} \
    --epoch_id {EPOCH_ID} \
    --num_timesteps 4 \
    --image_size 256 \
    --lr_size 16 \
    --num_channels 3 \
    --num_channels_dae 128 \
    --n_mlp 4 \
    --ch_mult 1 1 2 2 4 4 \
    --num_res_blocks 2 \
    --attn_resolutions 16 \
    --not_use_tanh \
    --batch_size 4 \
    --num_sample {NUM_SAMPLES}

print(f"✅ Results saved to {RESULTS_DIR}")

## Step 6 — Evaluate (PSNR / SSIM)

In [ ]:
!python benchmark/eval.py -p {RESULTS_DIR}

## Step 7 — Visualize Results

Display LR input / SR output / HR ground-truth side-by-side.

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

NUM_DISPLAY = 4   # number of image triplets to show

# Locate result images — adapt glob pattern to your test_srwddgan output layout
sr_paths = sorted(glob.glob(f"{RESULTS_DIR}/**/*fake*.png", recursive=True))[:NUM_DISPLAY]
hr_paths = sorted(glob.glob(f"{RESULTS_DIR}/**/*real*.png", recursive=True))[:NUM_DISPLAY]

if not sr_paths:
    # Fallback: look one level up
    sr_paths = sorted(glob.glob(f"{RESULTS_DIR}/*fake*.png"))[:NUM_DISPLAY]
    hr_paths = sorted(glob.glob(f"{RESULTS_DIR}/*real*.png"))[:NUM_DISPLAY]

if not sr_paths:
    print(f"No result images found in {RESULTS_DIR}. Check inference step output.")
else:
    fig, axes = plt.subplots(len(sr_paths), 2, figsize=(10, 5 * len(sr_paths)))
    if len(sr_paths) == 1:
        axes = [axes]

    for i, (sr_p, hr_p) in enumerate(zip(sr_paths, hr_paths)):
        sr_img = np.array(Image.open(sr_p))
        hr_img = np.array(Image.open(hr_p))
        axes[i][0].imshow(sr_img)
        axes[i][0].set_title("Super-Resolved (model output)", fontsize=11)
        axes[i][0].axis("off")
        axes[i][1].imshow(hr_img)
        axes[i][1].set_title("Ground Truth (HR)", fontsize=11)
        axes[i][1].axis("off")

    plt.tight_layout()
    plt.show()

## Step 8 — Save Checkpoints Back to Google Drive

In [ ]:
import shutil

DRIVE_CKPT_DIR = f"/content/drive/MyDrive/DeepSpace/checkpoints/{INFER_EXP}"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

LOCAL_CKPT_DIR = f"./content/saved_info/wddgan/{INFER_EXP}/{INFER_EXP}"

# Copy all .pth files to Drive
ckpts = glob.glob(f"{LOCAL_CKPT_DIR}/*.pth")
for ckpt in ckpts:
    dest = os.path.join(DRIVE_CKPT_DIR, os.path.basename(ckpt))
    shutil.copy2(ckpt, dest)
    print(f"  Saved: {os.path.basename(ckpt)}")

if not ckpts:
    print("No .pth checkpoints found — check LOCAL_CKPT_DIR path.")
else:
    print(f"\n✅ {len(ckpts)} checkpoint(s) saved to Drive: {DRIVE_CKPT_DIR}")